# Homework: Document Processing with AI

Books by Allen Downey (Green Tea Press) → extract text → chunk → index → RAG.

Run the cells top to bottom. Each step prints or displays something so you can check it before moving on. Answers to the homework questions are called out with **"ANSWER"** in the printed output.

## Step 0 — Install dependencies (uv)

Run once in a terminal, in the project folder:

```bash
uv init                     # only if this isn't a uv project yet

uv add requests pandas "markitdown[pdf]==0.1.7" minsearch==0.2.0 gitsource==0.0.5 openai pydantic python-dotenv
uv add --dev ipykernel
```

Then point the notebook kernel at this project's `.venv` (VS Code: *Select Kernel → Python Environments → .venv*), or run `uv run --with jupyter jupyter lab`.

In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version

expected = {
    "requests": None, "pandas": None, "markitdown": "0.1.7",
    "minsearch": "0.2.0", "gitsource": "0.0.5", "openai": None,
    "pydantic": None, "python-dotenv": None,
}

print("Python     :", sys.version.split()[0])
print("Interpreter:", sys.executable)
print()

all_ok = True
for package, wanted in expected.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    note = "NOT INSTALLED" if installed is None else (f"expected {wanted}" if wanted and installed != wanted else "ok")
    all_ok = all_ok and note == "ok"
    print(f"{package:<14} {installed or '-':<9} {note}")

print()
print("Environment ready." if all_ok else "Fix the packages above with `uv add ...`, restart the kernel, re-run.")

Python     : 3.10.12
Interpreter: /home/nima/code/nima-dx/ai-systems-lab/.venv/bin/python

requests       2.34.2    ok
pandas         2.3.3     ok
markitdown     0.1.7     ok
minsearch      0.2.0     ok
gitsource      0.0.5     ok
openai         3.16.2    ok
pydantic       2.13.5    ok
python-dotenv  1.2.4     ok

Environment ready.


## Step 1 — Download the books

Downloads `books.csv`, then every PDF it lists, into `books/`.

In [2]:
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from IPython.display import display

CSV_URL = (
    "https://raw.githubusercontent.com/alexeygrigorev/"
    "ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"
)
CSV_PATH = Path("books.csv")
BOOKS_DIR = Path("books")
TEXT_DIR = Path("books_text")

HEADERS = {"User-Agent": "Mozilla/5.0 (homework downloader)"}
TIMEOUT = 60

BOOKS_DIR.mkdir(exist_ok=True)
TEXT_DIR.mkdir(exist_ok=True)

response = requests.get(CSV_URL, headers=HEADERS, timeout=TIMEOUT)
response.raise_for_status()
CSV_PATH.write_bytes(response.content)

books = pd.read_csv(CSV_PATH)
books["filename"] = books["pdf_url"].apply(lambda url: Path(urlparse(url).path).name)

print(f"books.csv saved ({CSV_PATH.stat().st_size} bytes)")
with pd.option_context("display.max_colwidth", None):
    display(books[["title", "filename", "pdf_url"]])

books.csv saved (884 bytes)


,title,filename,pdf_url
0,Think Python 2e,thinkpython2.pdf,https://greenteapress.com/thinkpython2/thinkpython2.pdf
1,Think DSP,thinkdsp.pdf,https://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,thinkcomplexity2.pdf,https://greenteapress.com/complexity2/thinkcomplexity2.pdf
3,Think Java 2e,thinkjava2.pdf,https://greenteapress.com/thinkjava7/thinkjava2.pdf
4,Physical Modeling in MATLAB,PhysicalModelingInMatlab4.pdf,https://github.com/AllenDowney/PhysicalModelingInMatlab/raw/master/PhysicalModelingInMatlab4.pdf
5,Think OS,thinkos.pdf,https://greenteapress.com/thinkos/thinkos.pdf
6,Think C++,Think-C.pdf,https://raw.githubusercontent.com/tscheffl/ThinkC/refs/heads/master/PDF/Think-C.pdf


In [3]:
def is_pdf(path):
    with open(path, "rb") as f:
        return f.read(4) == b"%PDF"


def download_pdf(url, target, overwrite=False):
    if target.exists() and not overwrite and is_pdf(target):
        return "skipped (already there)"

    part = target.with_name(target.name + ".part")
    try:
        with requests.get(url, headers=HEADERS, timeout=TIMEOUT, stream=True) as r:
            r.raise_for_status()
            with open(part, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 256):
                    f.write(chunk)
        if not is_pdf(part):
            part.unlink()
            return "error: server did not return a PDF"
        part.replace(target)
        return "downloaded"
    except requests.RequestException as e:
        part.unlink(missing_ok=True)
        code = getattr(e.response, "status_code", None)
        return f"error: {type(e).__name__} ({f'HTTP {code}' if code else str(e)[:80]})"


statuses = []
for i, row in enumerate(books.itertuples(index=False), start=1):
    target = BOOKS_DIR / row.filename
    print(f"[{i}/{len(books)}] {row.title:<30} ", end="", flush=True)
    status = download_pdf(row.pdf_url, target)
    statuses.append(status)
    size = f"{target.stat().st_size / 1_048_576:6.2f} MB" if target.exists() else "   --    "
    print(f"{size}  {status}")

books["status"] = statuses
print()

pdf_files = sorted(BOOKS_DIR.glob("*.pdf"))
print(f"{len(pdf_files)} PDF files in {BOOKS_DIR.resolve()}:")
for p in pdf_files:
    print(f"  {p.name:<34} {p.stat().st_size:>12,} bytes   valid_pdf={is_pdf(p)}")

[1/7] Think Python 2e                  0.88 MB  skipped (already there)
[2/7] Think DSP                        4.08 MB  skipped (already there)
[3/7] Think Complexity 2e              6.00 MB  skipped (already there)
[4/7] Think Java 2e                    1.87 MB  skipped (already there)
[5/7] Physical Modeling in MATLAB      1.25 MB  skipped (already there)
[6/7] Think OS                         0.35 MB  skipped (already there)
[7/7] Think C++                        0.93 MB  skipped (already there)

7 PDF files in /home/nima/code/nima-dx/ai-systems-lab/src/books:
  PhysicalModelingInMatlab4.pdf         1,313,152 bytes   valid_pdf=True
  Think-C.pdf                             978,841 bytes   valid_pdf=True
  thinkcomplexity2.pdf                  6,294,093 bytes   valid_pdf=True
  thinkdsp.pdf                          4,275,659 bytes   valid_pdf=True
  thinkjava2.pdf                        1,957,646 bytes   valid_pdf=True
  thinkos.pdf                             368,105 bytes   valid_p

## Question 1 — Converting PDFs to Markdown

Converts every PDF in `books/` to a `.md` file in `books_text/`, using `markitdown`. This is slow — expect roughly 5–15 seconds per book.

In [4]:
import time

from markitdown import MarkItDown

converter = MarkItDown()
title_by_filename = dict(zip(books["filename"], books["title"]))

conversions = []
for i, pdf_path in enumerate(pdf_files, start=1):
    print(f"[{i}/{len(pdf_files)}] {pdf_path.name:<34} ", end="", flush=True)
    start = time.time()

    result = converter.convert(str(pdf_path))
    text = result.text_content

    md_path = TEXT_DIR / (pdf_path.stem + ".md")
    md_path.write_text(text, encoding="utf-8")

    conversions.append({
        "title": title_by_filename.get(pdf_path.name, pdf_path.stem),
        "pdf": pdf_path.name,
        "md_path": md_path.name,
        "characters": len(text),
    })

    elapsed = time.time() - start
    print(f"{elapsed:5.1f}s  -> {md_path.name}  ({len(text):,} chars)")

print()
conversions_df = pd.DataFrame(conversions)
with pd.option_context("display.max_colwidth", None):
    display(conversions_df)

md_files = sorted(TEXT_DIR.glob("*.md"))
print(f"{len(md_files)} markdown files in {TEXT_DIR.resolve()}")

[1/7] PhysicalModelingInMatlab4.pdf        4.1s  -> PhysicalModelingInMatlab4.md  (391,829 chars)
[2/7] Think-C.pdf                          2.6s  -> Think-C.md  (349,497 chars)
[3/7] thinkcomplexity2.pdf                 5.2s  -> thinkcomplexity2.md  (503,183 chars)
[4/7] thinkdsp.pdf                        10.6s  -> thinkdsp.md  (355,262 chars)
[5/7] thinkjava2.pdf                       7.2s  -> thinkjava2.md  (927,025 chars)
[6/7] thinkos.pdf                          1.7s  -> thinkos.md  (233,518 chars)
[7/7] thinkpython2.pdf                     5.4s  -> thinkpython2.md  (492,134 chars)



,title,pdf,md_path,characters
0,Physical Modeling in MATLAB,PhysicalModelingInMatlab4.pdf,PhysicalModelingInMatlab4.md,391829
1,Think C++,Think-C.pdf,Think-C.md,349497
2,Think Complexity 2e,thinkcomplexity2.pdf,thinkcomplexity2.md,503183
3,Think DSP,thinkdsp.pdf,thinkdsp.md,355262
4,Think Java 2e,thinkjava2.pdf,thinkjava2.md,927025
5,Think OS,thinkos.pdf,thinkos.md,233518
6,Think Python 2e,thinkpython2.pdf,thinkpython2.md,492134


7 markdown files in /home/nima/code/nima-dx/ai-systems-lab/src/books_text


## Question 2 — Chunking for RAG

**2a. Prepare documents** — for each `.md` file: split into lines, drop empty/whitespace-only lines, and build `{"source": filename, "content": [list of non-empty lines]}`.

In [5]:
def load_prepared_document(md_path):
    text = md_path.read_text(encoding="utf-8")
    lines = text.split("\n")
    non_empty_lines = [line for line in lines if line.strip()]
    return {
        "source": md_path.name,
        "content": non_empty_lines,
    }


prepared_docs = [load_prepared_document(p) for p in md_files]

rows = [
    {
        "title": title_by_filename.get(d["source"].replace(".md", ".pdf"), d["source"]),
        "source": d["source"],
        "non_empty_lines": len(d["content"]),
    }
    for d in prepared_docs
]
line_counts_df = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", None):
    display(line_counts_df)

print()
print("Example - first 5 non-empty lines of the first document:")
for line in prepared_docs[0]["content"][:5]:
    print(" ", repr(line))

,title,source,non_empty_lines
0,Physical Modeling in MATLAB,PhysicalModelingInMatlab4.md,5978
1,Think C++,Think-C.md,5371
2,Think Complexity 2e,thinkcomplexity2.md,7216
3,Think DSP,thinkdsp.md,5554
4,Think Java 2e,thinkjava2.md,12520
5,Think OS,thinkos.md,3444
6,Think Python 2e,thinkpython2.md,9461



Example - first 5 non-empty lines of the first document:
  '| Physical | Modeling | in MATLAB |'
  '| -------- | -------- | --------- |'
  'Version 4.0'
  'Allen B. Downey'
  'Green Tea Press'


**2b. Chunk** with `gitsource.chunk_documents`, `size=100`, `step=50`. Because `content` here is a *list of lines* (not a string), each chunk's `content` is a list of up to 100 lines, moved forward 50 lines at a time.

In [6]:
from gitsource import chunk_documents

CHUNK_SIZE = 100
CHUNK_STEP = 50

chunks = chunk_documents(prepared_docs, size=CHUNK_SIZE, step=CHUNK_STEP, content_field_name="content")

print(f"{len(prepared_docs)} documents -> {len(chunks)} chunks  (size={CHUNK_SIZE}, step={CHUNK_STEP})")
print()
print("Example chunk (metadata only, content truncated):")
example = chunks[5].copy()
example["content"] = example["content"][:3] + ["..."]
display(example)

7 documents -> 988 chunks  (size=100, step=50)

Example chunk (metadata only, content truncated):


{'start': 250,
 'content': ['everything, which can make it hard to get started. The organization of this book is meant to',
  'help: we start with simple numerical computations, adding vectors in Chapter 4 and matrices',
  '| in Chapter | 10. |     |     |     |     |     |     |     |     |',
  '...'],
 'source': 'PhysicalModelingInMatlab4.md'}

In [7]:
chunks_per_source = pd.Series([c["source"] for c in chunks]).value_counts()
chunks_per_book = chunks_per_source.rename(index=lambda s: title_by_filename.get(s.replace(".md", ".pdf"), s))

print("Chunks per book:")
display(chunks_per_book)

think_python_label = next(label for label in chunks_per_book.index if "think python" in label.lower())
tp_chunk_count = int(chunks_per_book[think_python_label])

print()
print(f"ANSWER (Q2) - chunks for '{think_python_label}': {tp_chunk_count}")

Chunks per book:


Think Java 2e                  250
Think Python 2e                189
Think Complexity 2e            144
Physical Modeling in MATLAB    119
Think DSP                      111
Think C++                      107
Think OS                        68
Name: count, dtype: int64


ANSWER (Q2) - chunks for 'Think Python 2e': 189


## Question 3 — Indexing with `minsearch`

Turn each chunk's `content` (a list of lines) into a single string with `"\n".join(...)`, then build and fit the index.

In [8]:
from minsearch import Index


def prepare_documents(chunks):
    prepared = []
    for chunk in chunks:
        copy = chunk.copy()
        copy["content"] = "\n".join(copy["content"])
        prepared.append(copy)
    return prepared


documents = prepare_documents(chunks)

index = Index(text_fields=["content"])
index.fit(documents)

print(f"ANSWER (Q3) - indexed documents (chunks): {len(documents)}")
print()
print("Example indexed document (content truncated to 300 chars):")
example = documents[5].copy()
example["content"] = example["content"][:300] + "..."
display(example)

ANSWER (Q3) - indexed documents (chunks): 988

Example indexed document (content truncated to 300 chars):


{'start': 250,
 'content': 'everything, which can make it hard to get started. The organization of this book is meant to\nhelp: we start with simple numerical computations, adding vectors in Chapter 4 and matrices\n| in Chapter | 10. |     |     |     |     |     |     |     |     |\n| ---------- | --- | --- | --- | --- | --- | -...',
 'source': 'PhysicalModelingInMatlab4.md'}

## Question 4 — Searching and RAG

Search the index and look at which book the top result came from.

## Question 5 — Full RAG

The exact `instructions` / `prompt_template` / `build_prompt` / `search` / `llm` / `rag` code from the lesson (so results are reproducible), using the OpenAI `responses` API.

Needs a real `OPENAI_API_KEY` (env var, or a `.env` file read by `python-dotenv`).

In [10]:
from dotenv import load_dotenv
load_dotenv()

import os
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


In [11]:
import json

from openai import OpenAI

openai_client = OpenAI()

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer


rag_query = "python function definition"
answer = rag(rag_query)

print("Question:", rag_query)
print()
print("Answer:")
print(answer)

Question: python function definition

Answer:
To define a function in Python, follow these steps:

1. **Use the `def` keyword**: This is the keyword that tells Python you're about to define a function.
2. **Name your function**: Follow the `def` keyword with the desired name of the function, adhering to Python's naming rules (it should start with a letter or underscore, followed by letters, numbers, or underscores).
3. **Add parentheses `()`**: Inside the parentheses, you define any input parameters your function will take.
4. **Use a colon `:`**: This indicates the start of the function body.
5. **Indent the function body**: The code that makes up the function must be indented.

Here’s an example of a simple function definition:

```python
def my_function(x):
    result = x * 2
    return result
```

### Breakdown:
- **`def my_function(x):`**: This defines a function named `my_function` that takes one parameter, `x`.
- **`result = x * 2`**: This line performs an operation (doubling `x

**Now track token usage.** The OpenAI `responses` API returns a `response.usage` object with `input_tokens` / `output_tokens` / `total_tokens`. `llm_with_usage` and `rag_with_usage` below are the same functions, just also returning that usage.

In [12]:
def llm_with_usage(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text, response.usage


def rag_with_usage(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, usage = llm_with_usage(prompt, instructions)
    return answer, usage


answer, usage = rag_with_usage(rag_query)

print("Answer:")
print(answer)
print()
print("Usage:", usage)
print()
print(f"ANSWER (Q5) - input tokens for this RAG query: {usage.input_tokens}")
print(f"(output tokens: {usage.output_tokens}, total: {usage.total_tokens})")

q5_input_tokens = usage.input_tokens

Answer:
To define a function in Python, you follow a specific syntax that includes the `def` keyword, the function name, parentheses for parameters, and a colon. The body of the function is indented below the definition line. Here’s how you can define a simple function:

```python
def my_function(x):
    # This function takes one input x and returns its square
    return x ** 2
```

### Key Points:
- **Defining a Function**: Use the `def` keyword followed by the function name and parameters.
- **Function Body**: The indented block after the function definition contains the code that runs when the function is called.
- **Return Statement**: Use `return` to output a value from the function.

### Example Usage:
Once you've defined the function, you can call it like this:

```python
result = my_function(5)
print(result)  # This will output 25
```

### Additional Information:
- **Function Inputs/Outputs**: You can specify multiple inputs separated by commas. For example: `def add(a, b): ret

## Question 6 — Structured vs Unstructured Output

Same RAG query, but the LLM call now uses `responses.parse` with a Pydantic `text_format`, so the answer comes back as a `RAGResponse` instead of free text.

In [13]:
from typing import Literal

from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")


def llm_structured(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse,
    )

    return response.output_parsed, response.usage


def rag_structured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    parsed, usage = llm_structured(prompt, instructions)
    return parsed, usage


parsed_answer, structured_usage = rag_structured(rag_query)

print("Structured answer:")
display(parsed_answer)
print()
print("Usage:", structured_usage)

Structured answer:


RAGResponse(answer='### Python Function Definition\nA function in Python is defined using the `def` keyword followed by the function name and parentheses containing any parameters. Below is the structure of a simple function:\n\n```python\ndef function_name(parameters):\n    # Code to execute\n    return result  # Optional\n```\n\n### Example\nHere is an example of a function that calculates the sum of two numbers:\n\n```python\ndef add_numbers(a, b):\n    result = a + b\n    return result\n```\n\n### Calling the Function\nTo call the function, you will use its name and pass the required arguments:\n\n```python\nsum_result = add_numbers(5, 3)  # sum_result will be 8\n```\n\n### Key Points\n- The function name should be descriptive of its action.\n- You can define multiple parameters separated by commas.\n- The `return` statement is optional; it specifies what value the function should output after execution.\n- Functions help avoid name collisions since each function has its own scope 


Usage: ResponseUsage(input_tokens=8112, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=350, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=8462)


In [14]:
q6_input_tokens = structured_usage.input_tokens
difference = q6_input_tokens - q5_input_tokens

print(f"Q5 (unstructured) input tokens: {q5_input_tokens}")
print(f"Q6 (structured)   input tokens: {q6_input_tokens}")
print()
print(f"ANSWER (Q6) - structured output uses {difference} MORE input tokens than unstructured.")

Q5 (unstructured) input tokens: 7926
Q6 (structured)   input tokens: 8112

ANSWER (Q6) - structured output uses 186 MORE input tokens than unstructured.
